# 01 — Data Engineering & Preprocessing (Amazon Reviews, UBCF)

Phase 1 (user-based adaptation):
- Chunked ingestion of Amazon Gift Cards 5-core JSON lines
- ID remapping and user-focused sparse matrix
- Truncated SVD (32 latent user features)
- Lightweight item content features for hybrid fallback

Outputs: `user_sparse.npz`, `user_latent.npy`, `content_features.npy`, `id_maps.pkl`

In [1]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz
from sklearn.decomposition import TruncatedSVD

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "AmazonReviews").exists() and (ROOT_DIR / "QACF" / "AmazonReviews").exists():
    ROOT_DIR = ROOT_DIR / "QACF"

DATA_PATH = ROOT_DIR / "AmazonReviews" / "Gift_Cards_5.json"
OUT_DIR = ROOT_DIR / "data" / "processed_amazon_5core"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 1_000_000
N_COMPONENTS = 32
SEED = 42

print(DATA_PATH)
print(f"Exists: {DATA_PATH.exists()}")
print(f"Output dir: {OUT_DIR}")

c:\Users\acolumban\Desktop\Quantum-Enhanced-Smart-Shopping-Experience\QACF\AmazonReviews\Gift_Cards_5.json
Exists: True
Output dir: c:\Users\acolumban\Desktop\Quantum-Enhanced-Smart-Shopping-Experience\QACF\data\processed_amazon_5core


In [2]:
# Chunked pass for ID discovery from Amazon JSON lines
all_users, all_items = set(), set()
n_rows = 0

for chunk in pd.read_json(DATA_PATH, lines=True, chunksize=CHUNK_SIZE):
    req = {"reviewerID", "asin", "overall"}
    if not req.issubset(chunk.columns):
        raise ValueError(f"Missing required columns in Amazon file. Expected {req}, got {set(chunk.columns)}")

    chunk = chunk[["reviewerID", "asin", "overall"]].dropna()
    chunk = chunk.rename(columns={"reviewerID": "user_raw", "asin": "item_raw", "overall": "rating"})

    n_rows += len(chunk)
    all_users.update(chunk["user_raw"].astype(str).unique().tolist())
    all_items.update(chunk["item_raw"].astype(str).unique().tolist())

orig_users = np.array(sorted(all_users), dtype=object)
orig_items = np.array(sorted(all_items), dtype=object)

user_to_idx = {u: i for i, u in enumerate(orig_users.tolist())}
item_to_idx = {m: i for i, m in enumerate(orig_items.tolist())}

# Keep numeric surrogate IDs for downstream notebook compatibility
unique_users = np.arange(len(orig_users), dtype=np.int64)
unique_movies = np.arange(len(orig_items), dtype=np.int64)

print(f"rows={n_rows:,} users={len(orig_users):,} items={len(orig_items):,}")

rows=2,972 users=458 items=148


In [3]:
# Build user-focused CSR matrix: rows=users, cols=items
row_parts, col_parts, val_parts = [], [], []
for chunk in pd.read_json(DATA_PATH, lines=True, chunksize=CHUNK_SIZE):
    chunk = chunk[["reviewerID", "asin", "overall"]].dropna()
    chunk = chunk.rename(columns={"reviewerID": "user_raw", "asin": "item_raw", "overall": "rating"})

    row_parts.append(chunk["user_raw"].astype(str).map(user_to_idx).to_numpy(dtype=np.int32, copy=False))
    col_parts.append(chunk["item_raw"].astype(str).map(item_to_idx).to_numpy(dtype=np.int32, copy=False))
    val_parts.append(chunk["rating"].to_numpy(dtype=np.float32, copy=False))

rows = np.concatenate(row_parts)
cols = np.concatenate(col_parts)
vals = np.concatenate(val_parts)
user_sparse = csr_matrix((vals, (rows, cols)), shape=(len(unique_users), len(unique_movies)), dtype=np.float32)

svd = TruncatedSVD(n_components=min(N_COMPONENTS, max(2, min(user_sparse.shape) - 1)), random_state=SEED)
user_latent = svd.fit_transform(user_sparse).astype(np.float32)

print(user_sparse.shape, user_sparse.nnz)
print(user_latent.shape)

(458, 148) 2965
(458, 32)


In [4]:
# Content features for hybrid cold-start fallback (Amazon items)
item_counts = np.asarray((user_sparse != 0).sum(axis=0)).ravel().astype(np.float32)
item_sum = np.asarray(user_sparse.sum(axis=0)).ravel().astype(np.float32)
item_mean = item_sum / np.maximum(item_counts, 1.0)

global_mean = float(user_sparse.data.mean()) if user_sparse.nnz else 0.0
item_mean[item_counts <= 0] = global_mean

log_pop = np.log1p(item_counts)
if np.max(log_pop) > 0:
    log_pop = log_pop / np.max(log_pop)

mean_centered = item_mean - np.mean(item_mean)
std_mean = np.std(mean_centered)
if std_mean > 1e-8:
    mean_centered = mean_centered / std_mean

# Deterministic hash bucket for sparse item identity signal
n_hash = 16
hash_buckets = np.array([hash(str(x)) % n_hash for x in orig_items], dtype=np.int32)
hash_onehot = np.zeros((len(orig_items), n_hash), dtype=np.float32)
hash_onehot[np.arange(len(orig_items)), hash_buckets] = 1.0

numeric_feats = np.stack([item_mean.astype(np.float32), log_pop.astype(np.float32), mean_centered.astype(np.float32)], axis=1)
content_features = np.hstack([numeric_feats, hash_onehot]).astype(np.float32)

save_npz(OUT_DIR / "user_sparse.npz", user_sparse)
np.save(OUT_DIR / "user_latent.npy", user_latent)
np.save(OUT_DIR / "content_features.npy", content_features)
with open(OUT_DIR / "id_maps.pkl", "wb") as f:
    pickle.dump(
        {
            "unique_users": unique_users,
            "unique_movies": unique_movies,
            "user_to_idx": user_to_idx,
            "movie_to_idx": item_to_idx,
            "orig_user_ids": orig_users,
            "orig_item_ids": orig_items,
        },
        f,
    )

print("Saved: user_sparse.npz, user_latent.npy, content_features.npy, id_maps.pkl")
print(f"content_features shape: {content_features.shape}")

Saved: user_sparse.npz, user_latent.npy, content_features.npy, id_maps.pkl
content_features shape: (148, 19)


## D-Wave Local Feature Selection (Simulator)

D-Wave annealing for feature selection in sparse user data (local sim now, real Leap later per supervisor request).

This section formulates a binary QUBO with one variable per latent feature ($z_i \in \{0,1\}$), where selected features maximize a variance-based proxy while enforcing a feature-budget penalty.

Objective (BQM proxy):
- Minimize $-\sum_i w_i z_i + \lambda(\sum_i z_i-k)^2$
- $w_i$: normalized variance contribution of feature $i$
- $k$: target number of selected latent features

In [ ]:
# D-Wave Local Feature Selection (Simulator)
from pathlib import Path
import numpy as np

use_dwave_sim = True

try:
    import dimod
except ImportError as exc:
    raise ImportError(
        "dimod is required for local D-Wave-style simulation. Install with: pip install dimod"
    ) from exc

if "OUT_DIR" not in globals():
    OUT_DIR = Path("data/processed_amazon_5core")
if "SEED" not in globals():
    SEED = 42

if "user_latent" not in globals():
    user_latent = np.load(OUT_DIR / "user_latent.npy")

X_latent = np.asarray(user_latent, dtype=np.float32)
n_users, n_features = X_latent.shape

# Target selected dimensionality (for 32 latent features, default keeps half)
TARGET_FEATURES = min(16, n_features)
TARGET_FEATURES = max(1, TARGET_FEATURES)

# Proxy objective weights: explained-variance approximation from per-feature variance
feature_var = X_latent.var(axis=0).astype(np.float64)
if np.allclose(feature_var.sum(), 0.0):
    feature_scores = np.ones(n_features, dtype=np.float64) / n_features
else:
    feature_scores = feature_var / feature_var.sum()

# Binary QUBO (BQM): minimize -variance_selected + cardinality penalty
k = int(TARGET_FEATURES)
lambda_cardinality = float(np.max(feature_scores) * 2.0 + 1e-9)

linear = {
    i: float(lambda_cardinality * (1 - 2 * k) - feature_scores[i])
    for i in range(n_features)
}
quadratic = {
    (i, j): float(2.0 * lambda_cardinality)
    for i in range(n_features)
    for j in range(i + 1, n_features)
}
offset = float(lambda_cardinality * (k ** 2))

bqm = dimod.BinaryQuadraticModel(linear, quadratic, offset, vartype=dimod.BINARY)

if use_dwave_sim:
    if n_features <= 20:
        sampler = dimod.ExactSolver()
        sample_set = sampler.sample(bqm)
    else:
        sampler = dimod.SimulatedAnnealingSampler()
        sample_set = sampler.sample(bqm, num_reads=50)
else:
    # Upgrade path for real D-Wave Leap later:
    # sampler = LeapHybridCQMSampler(token=os.getenv('DWAVE_API_TOKEN'))
    raise NotImplementedError("Set use_dwave_sim=True until D-Wave Leap token is available.")

best = sample_set.first.sample
selected_mask = np.array([1 if best[i] == 1 else 0 for i in range(n_features)], dtype=np.int8)

# Enforce exact-k fallback for stable downstream dimensions
if selected_mask.sum() != k:
    top_idx = np.argsort(feature_scores)[-k:]
    selected_mask[:] = 0
    selected_mask[top_idx] = 1

user_latent_selected = X_latent[:, selected_mask.astype(bool)]

np.save(OUT_DIR / "user_latent_selected.npy", user_latent_selected.astype(np.float32))
np.save(OUT_DIR / "selected_mask.npy", selected_mask)

retained_ratio = float(feature_var[selected_mask.astype(bool)].sum() / (feature_var.sum() + 1e-12))
print(f"D-Wave local sim complete | n_features={n_features}, selected={int(selected_mask.sum())}")
print(f"Approx retained variance ratio: {retained_ratio:.4f}")
print("Saved: user_latent_selected.npy, selected_mask.npy")

# Small-slice test (200 users) as quick sanity check
slice_n = min(200, n_users)
X_slice = X_latent[:slice_n]
X_slice_selected = X_slice[:, selected_mask.astype(bool)]
print(f"Slice test OK | input={X_slice.shape}, selected={X_slice_selected.shape}")

D-Wave local sim complete | n_features=32, selected=16
Approx retained variance ratio: 0.8269
Saved: user_latent_selected.npy, selected_mask.npy
Slice test OK | input=(200, 32), selected=(200, 16)
